# CO2SITE — Task A | Notebook A: Fictional learning example

> **How to derive scientifically justified indifference (q) and preference (p) thresholds for CO₂ storage capacity in PROMETHEE.**
>
> We aim to establish a reproducible procedure that can be applied to multiple CO₂ storage sites and assess whether it is sufficiently robust for implementation in the CO2SITE platform.

**This is a learning notebook with deliberately fictional capacity values.** We will do the same steps as the [GEUS-based research notebook (Notebook B)](https://colab.research.google.com/github/AnaSoles/ggg-co2-storage-eval/blob/research/task-a-capacity-indifference/examples/colab_task_a_capacity_q.ipynb), but with a smaller and easier dataset.

**How to use:** Runtime → Run all. Source code is collapsed by default; expand a cell to see how the results are produced. q estimates below are *exploratory illustrations*, not scientifically validated decision thresholds.

## 1. Understand each simulated capacity

Imagine that several Monte Carlo draws produced the following static storage capacities for two fictional sites:

| Draw | Fictional Site A (Mt) | Fictional Site B (Mt) | B minus A (Mt) |
|---|---:|---:|---:|
| 1 | 95 | 110 | +15 |
| 2 | 108 | 102 | −6 |
| 3 | 100 | 120 | +20 |
| 4 | 112 | 105 | −7 |
| 5 | 90 | 115 | +25 |

For this **teaching table only**, the five results represent *joint pairs* so we can subtract the values shown in the same row. There are 3 positive differences out of 5, so **P(B > A) = 60%** in this tiny fictional example. Five observations are not enough for a scientific q.

<details><summary>Show difference formula</summary>

$$\Delta_i=C_{B,i}-C_{A,i}$$

</details>

In [ ]:
# @title Calculate five fictional differences
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
a_small=np.array([95,108,100,112,90],dtype=float)
b_small=np.array([110,102,120,105,115],dtype=float)
diff_small=b_small-a_small
display(pd.DataFrame({"Draw":np.arange(1,6),"A (Mt)":a_small,"B (Mt)":b_small,"B − A (Mt)":diff_small}))
print(f"Probability in this 5-draw classroom example = {(diff_small>0).mean():.0%}")

## Figure 1. Visualise the five differences

Positive values mean Site B has more capacity *in that draw*. Negative values mean Site A has more.

In [ ]:
# @title Plot the simple differences
fig,ax=plt.subplots(figsize=(9,4))
ax.bar(np.arange(1,6),diff_small)
ax.axhline(0,color="black",linewidth=1)
ax.set(xlabel="Fictional draw number",ylabel="B minus A (Mt CO₂)",title="Figure 1 — Positive and negative differences")
ax.grid(axis="y",alpha=.2);plt.show()

## 2. Move to larger fictional Monte Carlo samples

For a more realistic statistical demonstration, let's **assume** that:
- Fictional Site A capacity follows a normal distribution with mean 100 Mt and SD 8 Mt.
- Fictional Site B capacity follows a normal distribution with mean 112 Mt and SD 9 Mt.
- The sites are independent, and the normal distributions are used *only* for this learning example.

Real geological capacity distributions often are not normal. The GEUS notebook uses PERT-based input assumptions and the original volumetric capacity calculation instead.

We independently sample the two model distributions and subtract B − A.

In [ ]:
# @title Simulate fictional site-capacity distributions
N=20_000
rng=np.random.default_rng(2026)
a=rng.normal(loc=100,scale=8,size=N)
b=rng.normal(loc=112,scale=9,size=N)
difference=b-a
display(pd.DataFrame({"Draw":np.arange(1,11),"A (Mt)":a[:10],"B (Mt)":b[:10],"B − A (Mt)":difference[:10]}).round(2))
print(f"Mean A={a.mean():.2f} Mt; SD A={a.std(ddof=1):.2f} Mt")
print(f"Mean B={b.mean():.2f} Mt; SD B={b.std(ddof=1):.2f} Mt")

## Figure 2. Fictional capacity distributions

Watch how the two distributions overlap. The overlap makes it possible for the lower-mean site to occasionally have a higher simulated capacity.

In [ ]:
# @title Plot site-capacity distributions
fig,ax=plt.subplots(figsize=(9,4.5))
ax.hist(a,bins=60,density=True,alpha=.55,label="Site A")
ax.hist(b,bins=60,density=True,alpha=.55,label="Site B")
ax.set(xlabel="Simulated capacity (Mt CO₂)",ylabel="Density",title="Figure 2 — Fictional capacity uncertainty")
ax.legend();ax.grid(alpha=.2);plt.show()

## 3. Difference distribution and probability

Now compare all the samples by taking the difference for each Monte Carlo iteration. The **sample standard deviation of the difference** tells us how dispersed these relative-capacity results are.

<details><summary>Show standard deviation formula (independent sites)</summary>

$$\sigma_{B-A}=\sqrt{\sigma_A^2+\sigma_B^2}$$

Here the assumed SDs give approximately **12.04 Mt**, before simulation.

</details>

A frequent positive difference does not, by itself, determine the PROMETHEE q.

In [ ]:
# @title Calculate probabilities and uncertainty interval
lower,upper=np.quantile(difference,[.025,.975])
display(pd.DataFrame({"Quantity":["Mean B−A","SD B−A","P(B>A)","2.5% quantile","97.5% quantile"],"Value":[f"{difference.mean():.2f} Mt",f"{difference.std(ddof=1):.2f} Mt",f"{(difference>0).mean():.1%}",f"{lower:.2f} Mt",f"{upper:.2f} Mt"]}))
fig,ax=plt.subplots(figsize=(9,4.5))
ax.hist(difference,bins=65)
ax.axvline(0,color="black",linestyle="--",label="Equal capacity")
ax.axvline(lower,linestyle=":",label="2.5%")
ax.axvline(upper,linestyle=":",label="97.5%")
ax.set(xlabel="B − A (Mt CO₂)",ylabel="Count",title="Figure 3 — Distribution of fictional differences")
ax.legend();ax.grid(alpha=.2);plt.show()

## 4. Explore a candidate q — and its limitation

As an **illustrative diagnostic only**, we take half the width of the central 95% interval of the difference distribution.

<details><summary>Show exploratory q formula</summary>

$$q^{\mathrm{exploratory}}_{AB}=\frac{Q_{97.5\%}(\Delta)-Q_{2.5\%}(\Delta)}{2}$$

</details>

This is a *spread measure* of simulated differences, not an automatically valid indifference threshold. q expresses a decision preference, so expert validation and sensitivity analysis are still required.

A very important distinction: this is a **pair-specific** statistic. Ordinary PROMETHEE typically uses a **common q for a given criterion**, so we must investigate whether such a shared rule makes sense for the full site portfolio.

In [ ]:
# @title Calculate exploratory q and plot preference curve
q_candidate=(upper-lower)/2
p_illustration=max(2*q_candidate,q_candidate+1) # Not a calibrated p!
print(f"Exploratory pair uncertainty width (candidate q): {q_candidate:.2f} Mt")
print(f"Illustrative p (NOT derived): {p_illustration:.2f} Mt")
x=np.linspace(0,p_illustration*1.5,400)
pref=np.where(x<=q_candidate,0,np.where(x>=p_illustration,1,(x-q_candidate)/(p_illustration-q_candidate)))
fig,ax=plt.subplots(figsize=(9,4.5))
ax.plot(x,pref,linewidth=2)
ax.axvline(q_candidate,linestyle="--",label="Candidate q")
ax.axvline(p_illustration,linestyle=":",label="Fictional p for illustration")
ax.set(xlabel="Capacity advantage of B over A (Mt CO₂)",ylabel="Single-criterion preference",ylim=(-.03,1.03),title="Figure 4 — PROMETHEE preference concept")
ax.grid(alpha=.2);ax.legend();plt.show()

## 5. What to take into the GEUS notebook

We now understand four different outputs:

1. **Capacity distribution:** what values the geological model permits under its assumptions.
2. **Capacity difference distribution:** what values B minus A may take.
3. **P(B > A):** how often the simulated B capacity exceeds the simulated A capacity.
4. **Candidate q:** an *extra decision rule* we are investigating; it does not follow automatically from Monte Carlo.

**Next:** open [Notebook B — GEUS-based cases](https://colab.research.google.com/github/AnaSoles/ggg-co2-storage-eval/blob/research/task-a-capacity-indifference/examples/colab_task_a_capacity_q.ipynb) and repeat the same questions with published geological input assumptions. The code and graph styles are intentionally similar to make comparisons easier.